
# 25A — V5 TG10 Chronology-Balanced Multitask Discovery

This is a **focused second-stage DEV test**, not a new broad feature search.

Why:
- 24B's raw nuisance metric strongly rewards chronology orientation imbalance.
- TG10 was the most stable astrology-only representation across waves.
- The broad Orthodox universe and ElasticNet were not stable enough.
- Axis heterogeneity may exist, but a production general score must not require knowing a future event's axis.

Two winner-eligible models only:

1. `TG10_SHARED_RIDGE_BALANCED`
2. `TG10_MULTITASK_EFFECT_RIDGE_BALANCED`

For model 2, axis-specific deviations are **training-only auxiliary parameters**.  
The deployed year score uses only the common global TG10 coefficients.

No Control. No CONFIRM. No new events. No new features.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from IPython.display import display

warnings.filterwarnings("ignore")
SEED=20260817
OUTER_REPEATS=5
OUTER_FOLDS=5
INNER_FOLDS=3
C_GRID=[0.003,0.01,0.03,0.10,0.30,1.00,3.00]
N_BOOTSTRAP=10000

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=repo_root()
PREV=ROOT/"research/ml/artifacts/v5_astrology_discovery_tournament"
PRIMARY=PREV/"V5_DISCOVERY_PAIR_DIFF_PRIMARY.csv"
BROAD=PREV/"V5_DISCOVERY_PAIR_DIFF_BROAD.csv"
DEC24=PREV/"V5_DISCOVERY_ASTROLOGY_TOURNAMENT_DECISION.json"
PREREG=ROOT/"research/ml_corpus/v5_ground_truth/V5_TG10_MULTITASK_BALANCED_PREREGISTRATION.json"
OUT=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced"
OUT.mkdir(parents=True,exist_ok=True)

for p in [PRIMARY,BROAD,DEC24,PREREG]:
    if not p.exists(): raise FileNotFoundError(p)

d24=json.load(open(DEC24,encoding="utf-8"))
pre=json.load(open(PREREG,encoding="utf-8"))
assert d24["status"]=="V5_NO_ASTROLOGY_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
assert d24["rules"]["production_control_scored"] is False
assert d24["rules"]["confirm_loaded_or_researched"] is False
assert pre["status"]=="PREDECLARED_AFTER_24B_DIAGNOSTICS_BEFORE_25A_REFIT"

primary=pd.read_csv(PRIMARY)
broad=pd.read_csv(BROAD)
TG10=sorted([c for c in primary.columns if c.startswith("diff__tg10__")])
assert len(TG10)==40, len(TG10)
assert set(TG10).issubset(broad.columns)

print("PRIMARY",len(primary),"pairs",primary.subject_id.nunique(),"subjects")
print("BROAD",len(broad),"pairs",broad.subject_id.nunique(),"subjects")
print("TG10 features",len(TG10))


PRIMARY 193 pairs 100 subjects
BROAD 197 pairs 104 subjects
TG10 features 40


## 1. Chronology composition

In [2]:

def composition(frame):
    rows=[]
    for dim in [None,"collection_wave","preassigned_axis"]:
        groups=[("TOTAL",frame)] if dim is None else list(frame.groupby(dim))
        for key,g in groups:
            rows.append({
                "dimension":"TOTAL" if dim is None else dim,
                "group":str(key),
                "n_pairs":len(g),
                "n_subjects":g.subject_id.nunique(),
                "positive_earlier_pair_share":g.positive_earlier_calc.mean(),
                "positive_earlier_subject_macro_share":
                    g.groupby("subject_id").positive_earlier_calc.mean().mean()
            })
    return pd.DataFrame(rows)

comp=composition(primary)
comp.to_csv(OUT/"V5_25A_CHRONOLOGY_COMPOSITION.csv",index=False)
display(comp)


,dimension,group,n_pairs,n_subjects,positive_earlier_pair_share,positive_earlier_subject_macro_share
0,TOTAL,TOTAL,193,100,0.621762,0.654798
1,collection_wave,E1,57,29,0.614035,0.658046
2,collection_wave,E2,23,18,0.782609,0.833333
3,collection_wave,ORIGINAL,113,53,0.592920,0.592385
4,preassigned_axis,COMPETITIVE,134,57,0.582090,0.563972
5,preassigned_axis,PROJECT,12,9,0.500000,0.537037
6,preassigned_axis,STATUS,47,34,0.765957,0.838235


## 2. Metric and subject-disjoint folds

In [3]:

def chronology_balanced_metric(frame,correct):
    z=frame[["subject_id","positive_earlier_calc"]].copy()
    z["correct"]=np.asarray(correct,dtype=float)
    vals=[]
    for ori in [0,1]:
        h=z[z.positive_earlier_calc==ori]
        if h.empty: return np.nan
        vals.append(h.groupby("subject_id").correct.mean().mean())
    return float(np.mean(vals))

def raw_subject_macro(frame,correct):
    z=frame[["subject_id"]].copy()
    z["correct"]=np.asarray(correct,dtype=float)
    return float(z.groupby("subject_id").correct.mean().mean())

def orientation_metrics(frame,correct):
    z=frame[["subject_id","positive_earlier_calc"]].copy()
    z["correct"]=np.asarray(correct,dtype=float)
    out={}
    for ori in [0,1]:
        h=z[z.positive_earlier_calc==ori]
        out[ori]=float(h.groupby("subject_id").correct.mean().mean()) if len(h) else np.nan
    return out

def make_subject_folds(frame,n_folds,seed):
    rng=np.random.RandomState(seed)
    axes=sorted(frame.preassigned_axis.unique())
    waves=sorted(frame.collection_wave.unique())
    rows=[]
    for sid,g in frame.groupby("subject_id"):
        r={"subject_id":str(sid),"n":len(g),
           "ori0":int((g.positive_earlier_calc==0).sum()),
           "ori1":int((g.positive_earlier_calc==1).sum())}
        for a in axes:r["axis__"+a]=int((g.preassigned_axis==a).sum())
        for w in waves:r["wave__"+str(w)]=int((g.collection_wave==w).sum())
        rows.append(r)
    m=pd.DataFrame(rows)
    m["jit"]=rng.uniform(size=len(m))
    m=m.sort_values(["n","jit"],ascending=[False,True]).reset_index(drop=True)
    cols=[c for c in m.columns if c not in {"subject_id","jit"}]
    target=m[cols].sum().to_numpy(float)/n_folds
    scale=np.maximum(target,1.0)
    sums=np.zeros((n_folds,len(cols))); folds=[[] for _ in range(n_folds)]
    for k in range(n_folds):
        row=m.iloc[k]; folds[k].append(row.subject_id)
        sums[k]+=row[cols].to_numpy(float)
    for i in range(n_folds,len(m)):
        row=m.iloc[i]; v=row[cols].to_numpy(float)
        costs=[]
        for k in range(n_folds):
            before=np.sum(((sums[k]-target)/scale)**2)
            after=np.sum(((sums[k]+v-target)/scale)**2)
            costs.append(after-before+1e-9*rng.uniform())
        k=int(np.argmin(costs))
        folds[k].append(row.subject_id); sums[k]+=v
    assigned=[s for f in folds for s in f]
    assert len(set(assigned))==frame.subject_id.nunique()
    return folds


## 3. Shared and multitask-effect Ridge models

In [4]:

AXIS_CODE={
    "COMPETITIVE":(1.0,0.0),
    "PROJECT":(0.0,1.0),
    "STATUS":(-1.0,-1.0),
}

def subject_pair_weights(frame):
    n=frame.groupby("subject_id").size()
    return frame.subject_id.map(lambda x:1.0/n.loc[x]).to_numpy(float)

def symmetrize(X,w):
    y=np.r_[np.ones(len(X),int),np.zeros(len(X),int)]
    return np.vstack([X,-X]),y,np.r_[w/2.0,w/2.0]

def fit_shared(train,C):
    scaler=StandardScaler().fit(train[TG10].to_numpy(float))
    X=scaler.transform(train[TG10].to_numpy(float))
    w=subject_pair_weights(train)
    X2,y,w2=symmetrize(X,w)
    clf=LogisticRegression(
        penalty="l2",C=float(C),solver="liblinear",
        fit_intercept=False,max_iter=5000,random_state=SEED
    ).fit(X2,y,sample_weight=w2)
    return scaler,clf

def score_shared(model,frame):
    scaler,clf=model
    X=scaler.transform(frame[TG10].to_numpy(float))
    return X@clf.coef_.ravel()

def design_multitask(Xstd,axes):
    e=np.asarray([AXIS_CODE[a] for a in axes],float)
    return np.hstack([Xstd, Xstd*e[:,[0]], Xstd*e[:,[1]]])

def fit_multitask(train,C):
    scaler=StandardScaler().fit(train[TG10].to_numpy(float))
    Xstd=scaler.transform(train[TG10].to_numpy(float))
    D=design_multitask(Xstd,train.preassigned_axis.astype(str).to_numpy())
    w=subject_pair_weights(train)
    D2,y,w2=symmetrize(D,w)
    clf=LogisticRegression(
        penalty="l2",C=float(C),solver="liblinear",
        fit_intercept=False,max_iter=5000,random_state=SEED
    ).fit(D2,y,sample_weight=w2)
    return scaler,clf

def score_multitask_general(model,frame):
    # Deployable score: global TG10 coefficients only; no axis needed.
    scaler,clf=model
    Xstd=scaler.transform(frame[TG10].to_numpy(float))
    p=len(TG10)
    beta_global=clf.coef_.ravel()[:p]
    return Xstd@beta_global

def score_multitask_task(model,frame):
    # Diagnostic only; uses known event axis.
    scaler,clf=model
    Xstd=scaler.transform(frame[TG10].to_numpy(float))
    D=design_multitask(Xstd,frame.preassigned_axis.astype(str).to_numpy())
    return D@clf.coef_.ravel()

def correct(score,eps=1e-12):
    s=np.asarray(score,float)
    return np.where(s>eps,1.0,np.where(s<-eps,0.0,0.5))

def tune_model(train,kind,seed):
    folds=make_subject_folds(train,min(INNER_FOLDS,train.subject_id.nunique()),seed)
    rows=[]
    for C in C_GRID:
        vals=[]
        for k,test_s in enumerate(folds):
            tr=train[~train.subject_id.astype(str).isin(test_s)]
            te=train[train.subject_id.astype(str).isin(test_s)]
            if kind=="shared":
                m=fit_shared(tr,C); sc=score_shared(m,te)
            else:
                m=fit_multitask(tr,C); sc=score_multitask_general(m,te)
            vals.append(chronology_balanced_metric(te,correct(sc)))
        rows.append({"C":C,"inner_balanced_macro":np.nanmean(vals),
                     "inner_p10":np.nanquantile(vals,.10)})
    grid=pd.DataFrame(rows).sort_values(
        ["inner_balanced_macro","inner_p10","C"],
        ascending=[False,False,True]
    ).reset_index(drop=True)
    return float(grid.iloc[0].C),grid


## 4. Repeated outer CV

In [5]:

pair_rows=[]; repeat_rows=[]; grid_rows=[]; fold_rows=[]

for repeat in range(OUTER_REPEATS):
    folds=make_subject_folds(primary,OUTER_FOLDS,SEED+1000*repeat)
    for fold,test_s in enumerate(folds):
        tr=primary[~primary.subject_id.astype(str).isin(test_s)].copy()
        te=primary[primary.subject_id.astype(str).isin(test_s)].copy()
        fold_rows.append({
            "repeat":repeat,"fold":fold,
            "n_test_pairs":len(te),"n_test_subjects":te.subject_id.nunique(),
            "positive_earlier_share":te.positive_earlier_calc.mean()
        })
        for kind,name in [
            ("shared","TG10_SHARED_RIDGE_BALANCED"),
            ("multitask","TG10_MULTITASK_EFFECT_RIDGE_BALANCED")
        ]:
            C,grid=tune_model(tr,kind,SEED+10000*repeat+fold)
            grid["repeat"]=repeat;grid["fold"]=fold;grid["model"]=name
            grid_rows.append(grid)
            if kind=="shared":
                m=fit_shared(tr,C)
                scores={"GENERAL":score_shared(m,te)}
            else:
                m=fit_multitask(tr,C)
                scores={
                    "GENERAL":score_multitask_general(m,te),
                    "TASK_CONDITIONED_DIAGNOSTIC":score_multitask_task(m,te)
                }
            for score_kind,sc in scores.items():
                cor=correct(sc)
                tmp=te[["pair_id","subject_id","positive_earlier_calc",
                        "preassigned_axis","collection_wave"]].copy()
                tmp["repeat"]=repeat;tmp["fold"]=fold
                tmp["model"]=name;tmp["score_kind"]=score_kind
                tmp["correct"]=cor
                pair_rows.append(tmp)

pair_oof=pd.concat(pair_rows,ignore_index=True)
grids=pd.concat(grid_rows,ignore_index=True)
folds=pd.DataFrame(fold_rows)
pair_oof.to_csv(OUT/"V5_25A_PRIMARY_OOF_PAIR_SCORES.csv",index=False)
grids.to_csv(OUT/"V5_25A_INNER_GRID.csv",index=False)
folds.to_csv(OUT/"V5_25A_OUTER_FOLD_BALANCE.csv",index=False)

rows=[]
for (model,kind,rep),g in pair_oof.groupby(["model","score_kind","repeat"]):
    ori=orientation_metrics(g,g.correct)
    rows.append({
        "model":model,"score_kind":kind,"repeat":rep,
        "raw_subject_macro":raw_subject_macro(g,g.correct),
        "chronology_balanced_macro":chronology_balanced_metric(g,g.correct),
        "positive_later_macro":ori[0],
        "positive_earlier_macro":ori[1]
    })
rep=pd.DataFrame(rows)
summary=rep.groupby(["model","score_kind"]).agg(
    raw_subject_macro=("raw_subject_macro","mean"),
    balanced_macro=("chronology_balanced_macro","mean"),
    balanced_std=("chronology_balanced_macro","std"),
    balanced_p10=("chronology_balanced_macro",lambda x:np.quantile(x,.10)),
    positive_later_macro=("positive_later_macro","mean"),
    positive_earlier_macro=("positive_earlier_macro","mean")
).reset_index().sort_values("balanced_macro",ascending=False)
summary.to_csv(OUT/"V5_25A_PRIMARY_LEADERBOARD.csv",index=False)
display(summary)


,model,score_kind,raw_subject_macro,balanced_macro,balanced_std,balanced_p10,positive_later_macro,positive_earlier_macro
0,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,GENERAL,0.659555,0.676772,0.021543,0.657928,0.723333,0.630211
1,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,TASK_CONDITIONED_DIAGNOSTIC,0.596412,0.611827,0.025368,0.583700,0.641333,0.582321
2,TG10_SHARED_RIDGE_BALANCED,GENERAL,0.564907,0.566152,0.010577,0.555620,0.571333,0.560970


## 5. Paired subject bootstrap: multitask general vs shared general

In [6]:

avg=pair_oof[pair_oof.score_kind=="GENERAL"].groupby(
    ["model","subject_id","positive_earlier_calc"]
).correct.mean().reset_index()
subjects=sorted(primary.subject_id.unique())

def matrix(model):
    m=np.full((len(subjects),2),np.nan)
    ix={s:i for i,s in enumerate(subjects)}
    for _,r in avg[avg.model==model].iterrows():
        m[ix[r.subject_id],int(r.positive_earlier_calc)]=r.correct
    return m

A=matrix("TG10_SHARED_RIDGE_BALANCED")
B=matrix("TG10_MULTITASK_EFFECT_RIDGE_BALANCED")

def metric_counts(M,counts):
    vals=[]
    for o in [0,1]:
        mask=~np.isnan(M[:,o]);den=counts[mask].sum()
        vals.append(np.sum(M[mask,o]*counts[mask])/den)
    return np.mean(vals)

rng=np.random.default_rng(SEED+999)
ds=[]
for _ in range(N_BOOTSTRAP):
    idx=rng.integers(0,len(subjects),len(subjects))
    counts=np.bincount(idx,minlength=len(subjects))
    ds.append(metric_counts(B,counts)-metric_counts(A,counts))
ds=np.asarray(ds)
boot=pd.DataFrame([{
    "comparison":"MULTITASK_GENERAL_minus_SHARED_GENERAL",
    "mean_delta":ds.mean(),
    "ci025":np.quantile(ds,.025),
    "ci975":np.quantile(ds,.975),
    "p_delta_gt_0":(ds>0).mean()
}])
boot.to_csv(OUT/"V5_25A_MULTITASK_VS_SHARED_BOOTSTRAP.csv",index=False)
display(boot)


,comparison,mean_delta,ci025,ci975,p_delta_gt_0
0,MULTITASK_GENERAL_minus_SHARED_GENERAL,0.110316,0.037571,0.184377,0.9985


## 6. BROAD fixed-algorithm sensitivity

In [7]:

def run_broad(frame):
    rows=[]
    for repeat in range(OUTER_REPEATS):
        folds=make_subject_folds(frame,OUTER_FOLDS,SEED+50000+1000*repeat)
        for fold,test_s in enumerate(folds):
            tr=frame[~frame.subject_id.astype(str).isin(test_s)]
            te=frame[frame.subject_id.astype(str).isin(test_s)]
            for kind,name in [
                ("shared","TG10_SHARED_RIDGE_BALANCED"),
                ("multitask","TG10_MULTITASK_EFFECT_RIDGE_BALANCED")
            ]:
                C,_=tune_model(tr,kind,SEED+60000+1000*repeat+fold)
                if kind=="shared":
                    m=fit_shared(tr,C);sc=score_shared(m,te)
                else:
                    m=fit_multitask(tr,C);sc=score_multitask_general(m,te)
                cor=correct(sc)
                tmp=te[["subject_id","positive_earlier_calc"]].copy()
                tmp["correct"]=cor;tmp["model"]=name;tmp["repeat"]=repeat
                rows.append(tmp)
    z=pd.concat(rows,ignore_index=True)
    sr=[]
    for (model,rep),g in z.groupby(["model","repeat"]):
        ori=orientation_metrics(g,g.correct)
        sr.append({
            "model":model,"repeat":rep,
            "balanced_macro":chronology_balanced_metric(g,g.correct),
            "positive_later_macro":ori[0],"positive_earlier_macro":ori[1]
        })
    rr=pd.DataFrame(sr)
    return rr.groupby("model").agg(
        balanced_macro=("balanced_macro","mean"),
        balanced_std=("balanced_macro","std"),
        balanced_p10=("balanced_macro",lambda x:np.quantile(x,.10)),
        positive_later_macro=("positive_later_macro","mean"),
        positive_earlier_macro=("positive_earlier_macro","mean")
    ).reset_index()

broad_summary=run_broad(broad)
broad_summary.to_csv(OUT/"V5_25A_BROAD_LEADERBOARD.csv",index=False)
display(broad_summary)


,model,balanced_macro,balanced_std,balanced_p10,positive_later_macro,positive_earlier_macro
0,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,0.636679,0.033011,0.602843,0.634,0.639357
1,TG10_SHARED_RIDGE_BALANCED,0.557602,0.043025,0.518044,0.564,0.551205


## 7. Wave/axis robustness for deployable GENERAL scores

In [8]:

avgpair=pair_oof[pair_oof.score_kind=="GENERAL"].groupby(
    ["model","pair_id","subject_id","positive_earlier_calc",
     "preassigned_axis","collection_wave"]
).correct.mean().reset_index()

def grouped_balanced(frame,dim):
    rows=[]
    for (model,key),g in frame.groupby(["model",dim]):
        ori=orientation_metrics(g,g.correct)
        rows.append({
            "model":model,dim:key,
            "n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
            "balanced_macro":chronology_balanced_metric(g,g.correct),
            "positive_later_macro":ori[0],"positive_earlier_macro":ori[1]
        })
    return pd.DataFrame(rows)

wave=grouped_balanced(avgpair,"collection_wave")
axis=grouped_balanced(avgpair,"preassigned_axis")
wave.to_csv(OUT/"V5_25A_WAVE_ROBUSTNESS.csv",index=False)
axis.to_csv(OUT/"V5_25A_AXIS_DIAGNOSTICS.csv",index=False)
display(wave);display(axis)


,model,collection_wave,n_pairs,n_subjects,balanced_macro,positive_later_macro,positive_earlier_macro
0,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,E1,57,29,0.721932,0.784444,0.659420
1,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,E2,23,18,0.617647,0.600000,0.635294
2,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,ORIGINAL,113,53,0.662051,0.713333,0.610769
3,TG10_SHARED_RIDGE_BALANCED,E1,57,29,0.557440,0.584444,0.530435
4,TG10_SHARED_RIDGE_BALANCED,E2,23,18,0.471765,0.320000,0.623529
5,TG10_SHARED_RIDGE_BALANCED,ORIGINAL,113,53,0.579188,0.606667,0.551709


,model,preassigned_axis,n_pairs,n_subjects,balanced_macro,positive_later_macro,positive_earlier_macro
0,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,COMPETITIVE,134,57,0.687654,0.726852,0.648455
1,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,PROJECT,12,9,0.540000,0.780000,0.300000
2,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,STATUS,47,34,0.673264,0.677778,0.668750
3,TG10_SHARED_RIDGE_BALANCED,COMPETITIVE,134,57,0.556177,0.604630,0.507724
4,TG10_SHARED_RIDGE_BALANCED,PROJECT,12,9,0.536667,0.440000,0.633333
5,TG10_SHARED_RIDGE_BALANCED,STATUS,47,34,0.563368,0.511111,0.615625


## 8. Frozen decision and optional candidate fit

In [9]:

L=summary[summary.score_kind=="GENERAL"].set_index("model")
BROAD=broad_summary.set_index("model")
BOOT=boot.iloc[0]

gate_rows=[]
for model in ["TG10_SHARED_RIDGE_BALANCED","TG10_MULTITASK_EFFECT_RIDGE_BALANCED"]:
    row=L.loc[model]
    w=wave[wave.model==model]
    supported=w[w.n_subjects>=10]
    wave_floor=float(supported.balanced_macro.min()) if len(supported) else np.nan
    passes={
      "primary_balanced_ge_055":row.balanced_macro>=0.55,
      "p10_ge_050":row.balanced_p10>=0.50,
      "positive_later_ge_050":row.positive_later_macro>=0.50,
      "positive_earlier_ge_050":row.positive_earlier_macro>=0.50,
      "broad_balanced_ge_053":BROAD.loc[model,"balanced_macro"]>=0.53,
      "wave_floor_ge_048":True if np.isnan(wave_floor) else wave_floor>=0.48,
    }
    if model=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED":
        passes["delta_vs_shared_ge_001"]=BOOT.mean_delta>=0.01
        passes["bootstrap_p_ge_080"]=BOOT.p_delta_gt_0>=0.80
    gate_rows.append({
      "model":model,
      "primary_balanced":row.balanced_macro,
      "primary_p10":row.balanced_p10,
      "positive_later":row.positive_later_macro,
      "positive_earlier":row.positive_earlier_macro,
      "broad_balanced":BROAD.loc[model,"balanced_macro"],
      "supported_wave_floor":wave_floor,
      **passes,
      "all_gates":all(passes.values())
    })

gates=pd.DataFrame(gate_rows)
gates.to_csv(OUT/"V5_25A_CANDIDATE_GATES.csv",index=False)
display(gates)

survivors=gates[gates.all_gates].copy()
winner=None
if len(survivors):
    # highest balanced macro; shared wins exact tie by simplicity
    survivors["simplicity"]=[
        0 if m=="TG10_SHARED_RIDGE_BALANCED" else 1
        for m in survivors.model
    ]
    survivors=survivors.sort_values(
        ["primary_balanced","simplicity"],ascending=[False,True]
    )
    winner=str(survivors.iloc[0].model)

if winner:
    status="V5_TG10_BALANCED_GENERAL_CANDIDATE_FROZEN_READY_FOR_CONTROL_BENCHMARK"
else:
    status="V5_TG10_BALANCED_MULTITASK_NO_CANDIDATE_SURVIVES"

# Freeze full-primary deployable global model if winner.
spec_path=OUT/"V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json"
coef_path=OUT/"V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv"

if winner:
    kind="shared" if winner=="TG10_SHARED_RIDGE_BALANCED" else "multitask"
    C,full_grid=tune_model(primary,kind,SEED+90000)
    if kind=="shared":
        scaler,clf=fit_shared(primary,C)
        beta=clf.coef_.ravel()
        deviations=None
    else:
        scaler,clf=fit_multitask(primary,C)
        p=len(TG10)
        beta=clf.coef_.ravel()[:p]
        deviations=clf.coef_.ravel()[p:]
    pd.DataFrame({
        "feature":TG10,
        "global_coefficient":beta,
        "scaler_mean":scaler.mean_,
        "scaler_scale":scaler.scale_
    }).to_csv(coef_path,index=False)
    spec={
      "version":"V5_25A_FROZEN_CANDIDATE_MODEL_SPEC_V1",
      "status":"FROZEN_BEFORE_CONTROL_AND_CONFIRM",
      "architecture":winner,
      "C":C,
      "features":TG10,
      "axis_required_at_inference":False,
      "multitask_axis_deviations_training_only":winner.startswith("TG10_MULTITASK"),
      "control_scored":False,
      "confirm_loaded":False,
      "coefficients_sha256":sha256_file(coef_path)
    }
    json.dump(spec,open(spec_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

decision={
 "version":"V5_25A_TG10_BALANCED_MULTITASK_DECISION_V1",
 "created_at":datetime.now().isoformat(timespec="seconds"),
 "status":status,
 "winner":winner,
 "primary_metric":"chronology_balanced_subject_macro_pairwise_accuracy",
 "gates":gates.to_dict(orient="records"),
 "bootstrap":boot.to_dict(orient="records"),
 "rules":{
   "new_feature_family_added":False,
   "manual_feature_selection":False,
   "axis_required_at_inference":False,
   "production_control_scored":False,
   "confirm_loaded_or_researched":False
 },
 "next_rule":(
   "If winner exists: benchmark frozen candidate against Production Control once with zero retuning. "
   "If no winner: keep CONFIRM sealed; do not run another broad feature search automatically."
 )
}
json.dump(decision,open(OUT/"V5_25A_TG10_BALANCED_MULTITASK_DECISION.json","w",encoding="utf-8"),
          ensure_ascii=False,indent=2)
print(json.dumps({"status":status,"winner":winner},ensure_ascii=False,indent=2))


,model,primary_balanced,primary_p10,positive_later,positive_earlier,broad_balanced,supported_wave_floor,primary_balanced_ge_055,p10_ge_050,positive_later_ge_050,positive_earlier_ge_050,broad_balanced_ge_053,wave_floor_ge_048,all_gates,delta_vs_shared_ge_001,bootstrap_p_ge_080
0,TG10_SHARED_RIDGE_BALANCED,0.566152,0.555620,0.571333,0.560970,0.557602,0.471765,True,True,True,True,True,False,False,NaN,NaN
1,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,0.676772,0.657928,0.723333,0.630211,0.636679,0.617647,True,True,True,True,True,True,True,True,True


{
  "status": "V5_TG10_BALANCED_GENERAL_CANDIDATE_FROZEN_READY_FOR_CONTROL_BENCHMARK",
  "winner": "TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
}



## Send back

After `Kernel Restart → Run All`, send:

```text
V5_25A_TG10_BALANCED_MULTITASK_DECISION.json
V5_25A_PRIMARY_LEADERBOARD.csv
V5_25A_CANDIDATE_GATES.csv
V5_25A_MULTITASK_VS_SHARED_BOOTSTRAP.csv
V5_25A_BROAD_LEADERBOARD.csv
V5_25A_WAVE_ROBUSTNESS.csv
V5_25A_AXIS_DIAGNOSTICS.csv
```

If a candidate freezes, also send:

```text
V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json
V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv
```

Do not score Control or open CONFIRM yourself.
